#### 교차검증과 최적의 파라미터 찾기
- 머신러닝을 사용할 때 모델의 정확도를 측정하기 위해 반듣시 사용해야 하는 방법
- 딥러닝시에는 데이터의 크기가 크므로 이 방법은 사용할 필요가 없다.

In [2]:
import pandas as pd

In [3]:
wine = pd.read_csv("../Data/wine.csv")
wine.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [4]:
wine.shape

(6497, 4)

#### Feature와 Target 분리

In [6]:
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

----
#### 검증 세트 추가
- 19번에서 훈련세트와 테스트세트만 가지고 작업을 하였지만 테스트 세트 작업에 파라미터를 조절하여 정확성을 높히면 실전 사용시 문제가 발생한다.
- 이런 과정을 방지하기 위해 훈련세트, 검증세트, 테스트세트로 구분하여 분석작업을 한다.

In [7]:
# 전체세트중 훈련세트와 테스트세트를 8:2로 분리
from sklearn.model_selection import train_test_split

In [8]:
train_input, test_input, train_target, test_target = \
    train_test_split(
        data,
        target,
        test_size=0.2,
        random_state=42
    )

In [9]:
# 훈련세트중 훈련세트와 검증세트를 8:2 기준으로 분리한다.
sub_input, val_input, sub_target, val_target = \
    train_test_split(
        train_input,
        train_target,
        test_size=0.2,
        random_state=42
    )

In [11]:
# 훈련세트, 검증세트, 테스트세트의 크기 구하기
print("Train : ", sub_input.shape)
print("Valid : ", val_input.shape)
print("Test  : ", test_input.shape)

Train :  (4157, 3)
Valid :  (1040, 3)
Test  :  (1300, 3)


In [ ]:
# 훈련세트와 검증세트로 결정트리 모델 만들기
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    min_impurity_decrease=0.0005
)
dt.fit(sub_input, sub_target)

print("Train : ", dt.score(sub_input, sub_target))
print("Valid : ", dt.score(val_input, val_target))

Train :  0.8946355544864084
Valid :  0.8653846153846154


In [14]:
# Test Set로 최종확인
print("Test : ", dt.score(test_input, test_target))

Test :  0.8638461538461538


----
#### 교차검증(Cross Validation)
- 교차검증의 한 파트를 폴드라고 하며 교차검증의 기본 Fold는 5이다.
- 훈련세트와 검증세트를 바꾸어 가며 정확도를 구하는 방법이다.
- 전체에 대한 정확도는 해당 값들의 평균으로 구한다.

In [15]:
from sklearn.model_selection import cross_validate

In [16]:
scores = cross_validate(dt, train_input, train_target)
scores

{'fit_time': array([0.00362015, 0.00316978, 0.00332713, 0.00371623, 0.00316191]),
 'score_time': array([0.00081062, 0.00068378, 0.00071001, 0.00072312, 0.00061274]),
 'test_score': array([0.86538462, 0.86923077, 0.8825794 , 0.84985563, 0.87102984])}

In [17]:
scores['test_score'].mean()

np.float64(0.8676160509365515)

----
#### Optuna
- 최적화 알고리즘 기반 탐색
- 정밀하고 효율적인 Hyper Parameter 탐색

In [ ]:
# !pip install optuna


   -------------------- ------------------- 2/4 [alembic]
   -------------------- ------------------- 2/4 [alembic]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ------------------------------ --------- 3/4 [optuna]
   ---------------------------------------- 4/4 [optuna]



In [20]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris

In [21]:
iris = load_iris()

train_input, test_input, train_target, test_target = \
    train_test_split(
        iris.data,
        iris.target,
        test_size=0.2,
        random_state=42,
        stratify=iris.target
    )

In [22]:
def find_param(trial):
    # 탐색할 하이퍼파라미터 정의
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 2, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 5)
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        random_state=42
    )
    return cross_val_score(clf, train_input, train_target, cv=5).mean()

In [23]:
study = optuna.create_study(direction='maximize')
study.optimize(find_param, n_trials=50)
print("최적의 파라미터 : ", study.best_params)

[I 2026-07-03 14:06:58,441] A new study created in memory with name: no-name-a6f0edae-770a-4419-850e-c767beb6fceb
[I 2026-07-03 14:06:59,003] Trial 0 finished with value: 0.95 and parameters: {'n_estimators': 107, 'max_depth': 6, 'min_samples_split': 7, 'min_samples_leaf': 2, 'bootstrap': True}. Best is trial 0 with value: 0.95.
[I 2026-07-03 14:06:59,385] Trial 1 finished with value: 0.9583333333333334 and parameters: {'n_estimators': 98, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 4, 'bootstrap': False}. Best is trial 1 with value: 0.9583333333333334.
[I 2026-07-03 14:06:59,762] Trial 2 finished with value: 0.9583333333333334 and parameters: {'n_estimators': 96, 'max_depth': 19, 'min_samples_split': 9, 'min_samples_leaf': 3, 'bootstrap': False}. Best is trial 1 with value: 0.9583333333333334.
[I 2026-07-03 14:07:00,329] Trial 3 finished with value: 0.95 and parameters: {'n_estimators': 109, 'max_depth': 11, 'min_samples_split': 4, 'min_samples_leaf': 4, 'bootstrap': T

최적의 파라미터 :  {'n_estimators': 98, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 4, 'bootstrap': False}


In [24]:
# 최종 모델 Hyper Parameter 조정하기
# 랜덤 포레스트 분류기
rf = RandomForestClassifier(
    n_estimators=98,
    max_depth=5,
    min_samples_split=2,
    min_samples_leaf=4,
    bootstrap=False,
    random_state=42
)

In [25]:
rf.fit(train_input, train_target)

,n_estimators,98
,criterion,'gini'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,4
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [26]:
print("Train : ", rf.score(train_input, train_target))
print("Test : ", rf.score(test_input, test_target))

Train :  0.9833333333333333
Test :  0.9666666666666667
